In [30]:
import polars as pl
import holidays

pl.Config.set_tbl_rows(24)
pl.Config.save()  # saves to ~/.config/polars/config.json

'{"environment":{"POLARS_ENGINE_AFFINITY":null,"POLARS_FMT_MAX_COLS":null,"POLARS_FMT_MAX_ROWS":"24","POLARS_FMT_NUM_DECIMAL":null,"POLARS_FMT_NUM_GROUP_SEPARATOR":null,"POLARS_FMT_NUM_LEN":null,"POLARS_FMT_STR_LEN":null,"POLARS_FMT_TABLE_CELL_ALIGNMENT":null,"POLARS_FMT_TABLE_CELL_LIST_LEN":null,"POLARS_FMT_TABLE_CELL_NUMERIC_ALIGNMENT":null,"POLARS_FMT_TABLE_DATAFRAME_SHAPE_BELOW":null,"POLARS_FMT_TABLE_FORMATTING":null,"POLARS_FMT_TABLE_HIDE_COLUMN_DATA_TYPES":null,"POLARS_FMT_TABLE_HIDE_COLUMN_NAMES":null,"POLARS_FMT_TABLE_HIDE_COLUMN_SEPARATOR":null,"POLARS_FMT_TABLE_HIDE_DATAFRAME_SHAPE_INFORMATION":null,"POLARS_FMT_TABLE_INLINE_COLUMN_DATA_TYPE":null,"POLARS_FMT_TABLE_ROUNDED_CORNERS":null,"POLARS_MAX_EXPR_DEPTH":null,"POLARS_STREAMING_CHUNK_SIZE":null,"POLARS_TABLE_WIDTH":null,"POLARS_VERBOSE":null,"POLARS_WARN_UNSTABLE":null},"direct":{"set_fmt_float":"mixed","set_float_precision":null,"set_thousands_separator":"","set_decimal_separator":".","set_trim_decimal_zeros":false}}'

In [31]:
lf = pl.scan_parquet(
    "../data/raw/yellow_tripdata_2025-01.parquet"
)  # lazy scan - loads on collect()
df = lf.collect()
df.columns = [c.lower() for c in df.columns]
df = df.rename(
    {
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "ratecodeid": "rate_code_id",
        "vendorid": "vendor_id",
        "pulocationid": "pickup_location_id",
        "dolocationid": "dropoff_location_id",
    }
)

In [32]:
# Load and join zones data
zones = pl.read_csv("../data/raw/taxi_zone_lookup.csv")
zones.columns = [c.lower() for c in zones.columns]
df = df.join(
    zones.rename({"locationid": "pickup_location_id",
                  "borough": "pickup_borough",
                  "zone": "pickup_zone",
                  "service_zone": "pickup_service_zone",
                  }),
    on="pickup_location_id",
    how="left",
)

df = df.join(
        zones.rename({"locationid": "dropoff_location_id",
                  "borough": "dropoff_borough",
                  "zone": "dropoff_zone",
                  "service_zone": "dropoff_service_zone",
                  }),
    on="dropoff_location_id",
    how="left",
)

In [33]:
# build is_airport_pickup field
df = df.with_columns(
    pl.col("pickup_location_id").is_in([1, 132, 138]).cast(pl.Int8).alias("is_airport_pickup")
)

In [34]:
# Build model columns
store_and_fwd_bin = {
    "N": 0,
    "Y": 1,
}

df = df.with_columns(
    ((pl.col("dropoff_datetime") - pl.col("pickup_datetime")).dt.total_seconds() / 60).alias(
        "duration_min"
    ),
    pl.col("store_and_fwd_flag")
    .replace_strict(store_and_fwd_bin, default=None)
    .alias("store_and_fwd_flag"),
    pl.col("pickup_datetime").dt.date().alias("pickup_date"),
    pl.col("pickup_datetime").dt.hour().alias("hour"),
    pl.col("pickup_datetime").dt.weekday().alias("day_of_week"),
    pl.col("pickup_datetime").dt.day().alias("day_of_month"),
).with_columns(
    pl.col("hour")
    .is_between(7, 10, closed="both")
    .or_(pl.col("hour").is_between(16, 19, closed="both"))
    .alias("is_rush_hour"),
)

In [35]:
# Adds holiday fields (is_holiday, distance_from_holiday, distance_to_holiday)

# Build holidays dict
holidays_dict = holidays.US(years=[2025])
holidays_list = list(holidays_dict.keys())

# Build a holidays frame
holidays_df = pl.DataFrame({
    "holiday_date": pl.Series(holidays_list).cast(pl.Date)
}).with_columns(pl.lit(1).alias("is_holiday"))

# Join to flag actual holidays
df = df.join(holidays_df, left_on="pickup_date", right_on="holiday_date", how="left").with_columns(
    pl.col("is_holiday").fill_null(0).cast(pl.Int8)
)
del holidays_df

## REACTIVATE WHEN WORKING MULTI-MONTH DATASET ##

# # Build a date-only reference frame for the asof joins (no is_holiday column)
# holiday_dates_df = pl.DataFrame({
#     "holiday_date": pl.Series(holidays_list).cast(pl.Date),
# })

# # Days since last holiday
# df = df.sort("pickup_date").join_asof(
#     holiday_dates_df.sort("holiday_date"),
#     left_on="pickup_date",
#     right_on="holiday_date",
#     strategy="backward"
# ).with_columns(
#     (pl.col("pickup_date") - pl.col("holiday_date"))
#     .dt.total_days()
#     .alias("days_since_holiday")
# ).drop("holiday_date")

# # Days to next holiday
# df = df.sort("pickup_date").join_asof(
#     holiday_dates_df.sort("holiday_date"),
#     left_on="pickup_date",
#     right_on="holiday_date",
#     strategy="forward"
# ).with_columns(
#     (pl.col("holiday_date") - pl.col("pickup_date"))
#     .dt.total_days()
#     .alias("days_to_holiday")
# ).drop("holiday_date")

# display(df)

In [36]:
# %%script echo skipping
# print("This code will be completely ignored.")
# EDA ONLY — drop before modeling
# Build readability columns
vendor_id_map = {
    1: "Creative Mobile Technologies LLC",
    2: "Curb Mobility, LLC",
    6: "Myle Technologies Inc",
    7: "Helix",
}

payment_type_map = {
    0: "flex_fare_trip",
    1: "credit_card",
    2: "cash",
    3: "no_charge",
    4: "dispute",
    5: "unknown",
    6: "voided_trip",
}


rate_code_map = {
    1: "standard_rate",
    2: "jfk",
    3: "newark",
    4: "nassau_or_winchester",
    5: "negotiated_fare",
    6: "group_ride",
    99: "null_or_unknown",
}

df = df.with_columns(
    # Create a new column with readable payment type
    (
        pl.col("payment_type")
        .replace_strict(payment_type_map, default=None)
        .alias("payment_type_str")
    ),
    # Create a new column with readable rate code
    (pl.col("rate_code_id").replace_strict(rate_code_map, default=None).alias("rate_code_str")),
    # Create a new column with readable vendor ID
    (pl.col("vendor_id").replace_strict(vendor_id_map, default=None).alias("vendor_id_str")),
)

In [37]:
# Add avg mph field
df = df.with_columns(
    (pl.col("trip_distance") / (pl.col("duration_min") / 60)).alias("avg_speed_mph")
)
df_raw = df.select(df)

In [38]:
# Condence borough mapping due to signal largely differing between 4 boroughs

borough_map = {
    "Manhattan": "manhattan",
    "Queens": "queens",
    "Bronx": "bronx",
    "Brooklyn": "brooklyn",
}

df = df.with_columns(
    pl.col("pickup_borough")
      .replace_strict(borough_map, default="other")
      .alias("pickup_borough"),
    pl.col("dropoff_borough")
      .replace_strict(borough_map, default="other")
      .alias("dropoff_borough"),
)

In [39]:
df.schema

Schema([('vendor_id', Int32),
        ('pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('rate_code_id', Int64),
        ('store_and_fwd_flag', Int64),
        ('pickup_location_id', Int32),
        ('dropoff_location_id', Int32),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('airport_fee', Float64),
        ('cbd_congestion_fee', Float64),
        ('pickup_borough', String),
        ('pickup_zone', String),
        ('pickup_service_zone', String),
        ('dropoff_borough', String),
        ('dropoff_zone', String),
        ('dropoff_service_zone', 

In [40]:
df.describe()

statistic,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,pickup_borough,pickup_zone,pickup_service_zone,dropoff_borough,dropoff_zone,dropoff_service_zone,is_airport_pickup,duration_min,pickup_date,hour,day_of_week,day_of_month,is_rush_hour,is_holiday,payment_type_str,rate_code_str,vendor_id_str,avg_speed_mph
str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,f64,f64,str,f64,f64,f64,f64,f64,str,str,str,f64
"""count""",3.475226e6,"""3475226""","""3475226""",2.935077e6,3.475226e6,2.935077e6,2.935077e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,2.935077e6,2.935077e6,3.475226e6,"""3475226""","""3475226""","""3475226""","""3475226""","""3475226""","""3475226""",3.475226e6,3.475226e6,"""3475226""",3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,"""3475226""","""2935077""","""3475226""",3.475226e6
"""null_count""",0.0,"""0""","""0""",540149.0,0.0,540149.0,540149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,540149.0,540149.0,0.0,"""0""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,"""0""",0.0,0.0,0.0,0.0,0.0,"""0""","""540149""","""0""",0.0
"""mean""",1.785428,"""2025-01-17 11:02:55.910964""","""2025-01-17 11:17:56.997901""",1.297859,5.855126,2.482535,0.002605,165.191576,164.125177,1.036623,17.081803,1.317737,0.478099,2.959813,0.449308,0.954795,25.611292,2.225237,0.123911,0.483409,null,null,null,null,null,null,0.067959,15.018116,"""2025-01-16 20:12:02.190729""",14.346836,4.049138,16.84187,0.430088,0.051876,null,null,null,NaN
"""std""",0.426328,null,null,0.75075,564.6016,11.632772,0.050973,64.529483,69.401686,0.701333,463.472918,1.861509,0.137462,3.779681,2.002582,0.278194,463.658478,0.903993,0.472509,0.361931,null,null,null,null,null,null,0.251675,38.713582,null,5.802688,1.840703,8.721751,null,0.221777,null,null,null,NaN
"""min""",1.0,"""2024-12-31 20:47:55""","""2024-12-18 07:52:40""",0.0,0.0,1.0,0.0,1.0,1.0,0.0,-900.0,-7.5,-0.5,-86.0,-126.94,-1.0,-901.0,-2.5,-1.75,-0.75,"""bronx""","""Allerton/Pelham Gardens""","""Airports""","""bronx""","""Allerton/Pelham Gardens""","""Airports""",0.0,-51472.316667,"""2024-12-31""",0.0,1.0,1.0,0.0,0.0,"""cash""","""group_ride""","""Creative Mobile Technologies L…",-64368.0
"""25%""",2.0,"""2025-01-10 07:59:01""","""2025-01-10 08:15:29""",1.0,0.98,1.0,0.0,132.0,113.0,1.0,8.6,0.0,0.5,0.0,0.0,1.0,15.2,2.5,0.0,0.0,null,null,null,null,null,null,0.0,7.283333,"""2025-01-10""",11.0,3.0,10.0,null,0.0,null,null,null,7.2103
"""50%""",2.0,"""2025-01-17 15:41:34""","""2025-01-17 15:59:34""",1.0,1.67,1.0,0.0,162.0,162.0,1.0,12.11,0.0,0.5,2.45,0.0,1.0,19.95,2.5,0.0,0.75,null,null,null,null,null,null,0.0,11.7,"""2025-01-17""",15.0,4.0,17.0,null,0.0,null,null,null,9.520343
"""75%""",2.0,"""2025-01-24 19:34:06""","""2025-01-24 19:48:31""",1.0,3.1,1.0,0.0,234.0,234.0,1.0,19.5,2.5,0.5,3.93,0.0,1.0,27.78,2.5,0.0,0.75,null,null,null,null,null,null,0.0,18.333333,"""2025-01-24""",19.0,6.0,24.0,null,0.0,null,null,null,12.941176
"""max""",7.0,"""2025-02-01 00:00:44""","""2025-02-01 23:44:11""",9.0,276423.57,99.0,1.0,265.0,265.0,5.0,863372.12,15.0,10.5,400.0,170.94,1.0,863380.37,2.5,6.75,0.75,"""queens""","""Yorkville West""","""Yellow Zone""","""queens""","""Yorkville West""","""Yellow Zone""",1.0,5626.316667,"""2025-02-01""",23.0,7.0,31.0,1.0,1.0,"""unknown""","""standard_rate""","""Myle Technologies Inc""",inf


In [41]:
# Null-value handling in important columns

# Define per-column replacement values for keep fields that have null values in them
null_replacement_map = {
    # Default to 1 passenger to assume that every ride has at least one passenger
    # Median non-zero passenger count is also 1
    "passenger_count": 1,
    # Most common value for store_and_fwd_flag is 0
    # Assume normal behavior for null values
    "store_and_fwd_flag": 0,
}

# Apply the dynamic fill_null operation
df = df.with_columns([pl.col(col).fill_null(val) for col, val in null_replacement_map.items()])

In [42]:
# Cleaning Calls!

keep_list = [
    "hour",
    "day_of_week",
    "is_rush_hour",
    "pickup_borough",
    "dropoff_borough",
    "is_airport_pickup",
    "duration_min",
]
df = df.filter(
    # Remove durations less than 1 minute as these are more likely to be errors/noise
    pl.col("duration_min") > 1,
    # Remove trips with invalid pickups prior to the implementation of the congestion fee on 2025-01-05
    # Remove trips with dates outside of January 2025
    pl.col("pickup_datetime").is_between(
        pl.datetime(2025, 1, 5), pl.datetime(2025, 1, 31, 23, 59, 59), closed="none"
    ),
    # Assume any trips where the average speed is outside of normal range for the operating area have errors in readings
    pl.col("avg_speed_mph").is_between(1, 85),
    # Vendor IDs 6 and 7 have a high percentage of records with meter/GPS errors, i.e. >95% of trips have duration of 0 minutes.
    # Remove vendor IDs 6 and 7 for consistency and data quality purposes
    pl.col("vendor_id").is_in([1, 2]),
)

df = df.select(keep_list)

In [43]:
print(f"Rows before cleaning: {len(df_raw)}")
print(f"Rows after cleaning: {len(df)}")
print(f"Retained: {len(df) / len(df_raw):.1%}")

df.write_parquet("../data/processed/taxi_2025-01_features.parquet")

Rows before cleaning: 3475226
Rows after cleaning: 3000627
Retained: 86.3%


# EDA Notes

### Fields Known at Pickup:

- **VendorID** - I assume this has to be the taxi proprietor
- RatecodeID
- **tpep_pickup_datetime** - goes without saying
- **PULocationID** - again, you should know where you are
- **is_airport_ride** - Derived from PULocationID

### Pickup/Dropoff

- Earliest pickup datetime is after the earliest dropoff datetime
  - Going to need to remove records where:
    - tpep_dropoff_datetime <= tpep_pickup_datetime

### Zero Values

- **Trip Distance:** 90,893 zero distance records
- **Passenger Count:** 24,656 zero passenger records
- **Fare Amount:** 1,398 zero fare amount records (May or may not be legitimate)
  - Small population suggests that this is an error rather than legit zeros.
- **Total Amount:** 559 zero total amount records (May or may not be legitimate)
  - Small population suggests that this is an error rather than legit zeros.
  - _NOTE:_ Zero total amount population is smaller than zero fare amount population./nWarrants further investigation as this could be either:
    - The ride was actually free and the individual provided a tip or there was a categorical surcharge present
    - 559 of the 1,398 zero fare amount records should have a larger value present and the remainder were legitimate zero fare rides
- **Trip Duration:** Only 1,927 zero minute duration rides is vast difference from the 90,893 zero trip distance records
  - Could be cab starts meter after stopping for pickup and then ride is cancelled with time charged?
    - I don't live in NY, so I'm not sure how this works.

### Tail Analysis

- Trip distance has massive max error tail
  - p99.9 is only 28.86, so 276423.57 is both, per the dataset and mathematically, impossible
- Passenger count has only 18 records > 6.0 (four 7.0 records, eleven 8.0 records, and three 9.0 records)
  - Given there are millions of records, it seems wiser to just cut these out, although it is technically possible that there may have been babies present with parents or people sitting on laps illegally, just saying.
- Any negative trip duration/trip distance values can be removed from the dataset.

### Cross-Column Relationship Issues

- **Trip Duration x Trip Distance**
  - Potential problem in terms of its error tail. It's hard to say where the errors are and where some of them are just truly long trips when cross referenced with **Trip Distance.** There is a 91 mile trip where the meter ran for 5000+ minutes. Given that that is 3.5 days of usage, I doubt the time is entirely legitimate, but it is a long haul trip for sure. There are also hundreds where it ran for 300+ minutes on a 15-20 mile trip. So it's difficult to pin down where to actually remove meter errors and which are legitimate fares/trip durations.
  - Duration should have a minimum cutoff as well in relation to distance of some sort. Trips less than 1 minute seem ridiculous logically, but also, if something has a long distance then there needs to be a logical amount of trip duration for that distance.
  - _NOTE:_ Perhaps get trimmed, weighted average for time per mile travelled to determine a reasonable min/max for the cross-column relationship and remove anything that falls outside of that. I assume there are going to be accuracy issues as part of this. Could also create some sort of likelihood value of the mile/minute value and then filter based on that?
